# V768 - Measuring Vision with Psychophysics

## Lab 8 - Motion Adaptation

In this lab, we will use a technique called top-up adaptation to measure the perceptual effects of motion adaptation.

### Learning Outcomes

(i.e., what you will be able to do at the end of this lab)

- Build a simple experiment using PsychoPy.

- Explain the impact of motion adaptation on the perception of motion direction in a motion direction discrimination task.

### Questions

- Explain what top-up adaptation is.

- Include and write a caption for the LAST figure in Part A, Step 3 (Analyze the data).

- Include and write a caption for the LAST figure in Part B, Step 3 (Analyze the data).

- Compare the results from both experiments. What is similar? What is different? If there are differences, explain what caused them.

## Part A. Simple Motion Coherence Experiment

### Step 1. Data collection

To collect data, open the PsychoPy ".psyexp" file in the folder titled "motion-simple-example", and run that experiment. The instructions will appear on the screen before the trials begin. A single run of the experiment includes 140 trials.

### Step 2. Load your data

#### Find files for specific participant ID

Before running this section of code, enter the participant ID you used below.


In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

rng = np.random.default_rng(2025)


def normcdf(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    sigma = max(float(sigma), np.finfo(float).eps)
    return norm.cdf(x, loc=float(mu), scale=sigma)


def norminv(p, mu, sigma):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, np.finfo(float).eps, 1 - np.finfo(float).eps)
    sigma = max(float(sigma), np.finfo(float).eps)
    return norm.ppf(p, loc=float(mu), scale=sigma)


def psyfxn(c, x):
    c = np.asarray(c, dtype=float)
    return (normcdf(x, c[0], c[1]) * (1 - c[2] - c[3])) + c[2]


def ipsyfxn(c, p):
    c = np.asarray(c, dtype=float)
    return norminv(
        (np.asarray(p, dtype=float) - c[2]) / (1 - c[2] - c[3]),
        c[0],
        c[1],
    )


def neg_log_likelihood(params, x, k, n):
    """Binomial negative log-likelihood used by the MATLAB teaching code."""
    x = np.asarray(x, dtype=float)
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)
    p = np.clip(psyfxn(params, x), 1e-5, 1 - 1e-5)
    return -np.sum(k * np.log(p) + (n - k) * np.log(1 - p))


def fit_psychometric_scipy(x, k, n, initial_params, bounds, method="L-BFGS-B"):
    """Return bounded maximum-likelihood point estimates using SciPy."""
    if method == "SLSQP":
        options = {"maxiter": 10_000, "ftol": 1e-12}
    else:
        options = {"maxiter": 10_000, "ftol": 1e-12, "gtol": 1e-8}
    result = minimize(
        neg_log_likelihood,
        x0=np.asarray(initial_params, dtype=float),
        args=(x, k, n),
        method=method,
        bounds=bounds,
        options=options,
    )
    if not result.success:
        raise RuntimeError(f"SciPy fit did not converge: {result.message}")
    return result


def find_participant_files(data_dir, participant_id):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            "PsychoPy data folder not found. Try changing your current Python "
            "folder to the folder containing this notebook."
        )
    file_paths = sorted(path for path in data_dir.glob(f"{participant_id}*.csv") if path.name.startswith(participant_id))
    if not file_paths:
        raise FileNotFoundError("No files found.")
    file_info = pd.DataFrame({
        "name": [path.name for path in file_paths],
        "folder": [str(path.parent) for path in file_paths],
        "path": file_paths,
    })
    print("data files found:")
    for name in file_info["name"]:
        print(f'    "{name}"')
    return file_info


def load_psychopy_csvs(file_info, remove_instruction_rows=True):
    frames = []
    for path in file_info["path"]:
        this_data = pd.read_csv(path)
        if remove_instruction_rows:
            if "thisN" in this_data.columns:
                this_data = this_data[this_data["thisN"].notna()]
            elif "trials.thisN" in this_data.columns:
                this_data = this_data[this_data["trials.thisN"].notna()]
            else:
                this_data = this_data.iloc[1:]
        frames.append(this_data.reset_index(drop=True))
    data = pd.concat(frames, ignore_index=True)
    data = data.drop(columns=["notes", "begin_experiment.started", "begin_experiment.stopped"], errors="ignore")
    return data


def display_fit_table(params, columns=("fit",)):
    return pd.DataFrame(
        np.asarray(params).reshape(4, -1),
        index=["mu", "sigma", "gamma", "lambda"],
        columns=list(columns),
    )
participant_id = "demo"

# check current working directory and find files
file_info = find_participant_files(Path("motion-simple-example") / "data", participant_id)
n_files = len(file_info)


**Confirm that your data file is listed above.** There should be one file.

#### Load and merge data tables


In [ ]:
# load data and stack it into one table
expt_data = load_psychopy_csvs(file_info)

# display data table size
n_trials = len(expt_data)
print(f"number of trials = {n_trials}")


### Step 3. Analyze your data

For this experiment, the stimulus was a field of dots. Both direction of motion and motion coherence varied. The coherence defines the proportion of dots that move in the direction of motion. The direction of motion was either to the right (0 degrees) or to the left (180 degrees), and the task was to report whether the motion was rightward or leftward.

For each trial, PsychoPy saved the motion direction, the motion coherence, and the participant's response. The responses were stored as 'left' or 'right'.


In [ ]:
# stash the stimulus location and response data
motion_coherence = expt_data["coherence"].astype(float)
motion_direction = expt_data["direction"].astype(float).copy()
resp_key = expt_data["key_resp.keys"].astype(str)

# convert stimulus motion direction to 1 for right and -1 for left
motion_direction = motion_direction.replace({180: -1, 0: 1})

# calculate signed motion coherence
signed_motion_coherence = motion_coherence * motion_direction

# convert the response key to 1 for right and -1 for left
# (to match stimulus direction)
resp_direction = np.where(resp_key.eq("right"), 1, -1)


#### Determine whether responses were correct or incorrect

For a response to be correct, the stimulus motion and response directions must match.


In [ ]:
# determine whether responses were correct or incorrect
resp_correct = motion_direction.to_numpy() == resp_direction

# determine whether participant chose right
chose_right = resp_direction == 1

# display overall performance
print(f"overall percentage correct = {np.mean(resp_correct) * 100:.2f}%")


#### Determine performance per condition

We need to now separate responses by stimulus condition to visualize performance in a psychometric curve. To accomplish that, we can separate by the signed motion coherence values that we calculated above. (Positive values indicate rightward motion, and negative values indicate leftward motion.)


In [ ]:
# average across responses per contrast level
analysis_table = pd.DataFrame({"response": chose_right, "coherence": signed_motion_coherence})
expt_results = (
    analysis_table
    .groupby("coherence", as_index=False)
    .agg(n_trials=("response", "size"), prob_right=("response", "mean"))
)

# stash stimulus levels & counts
coherence_levels = np.sort(expt_results["coherence"].unique())
n_coherence = len(coherence_levels)

# display results
display(expt_results)


#### Plot performance per condition


In [ ]:
fig, ax = plt.subplots()
ax.scatter(expt_results["coherence"], expt_results["prob_right"], color='k', linewidth=2)
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
plt.show()


#### **Fit data.**

Use SciPy to minimize the same binomial negative log-likelihood as the MATLAB `fmincon` code. The following starting values and bounds reproduce the MATLAB teaching model and return maximum-likelihood point estimates without priors.


In [ ]:
expt_scipy_result = fit_psychometric_scipy(
    x=expt_results["coherence"].to_numpy(dtype=float),
    k=(expt_results["prob_right"] * expt_results["n_trials"]).to_numpy(dtype=float),
    n=expt_results["n_trials"].to_numpy(dtype=float),
    initial_params=[0, 5, 0.1, 0.1],
    bounds=[(0, 180), (0, 100), (0, 0.5), (0, 0.5)],
)
expt_params = expt_scipy_result.x
discrim_param_table = display_fit_table(expt_params)
display(discrim_param_table)
print(f"negative log-likelihood = {expt_scipy_result.fun:.6f}")


#### Plot fit with data.


In [ ]:
fig, ax = plt.subplots()
xx = np.arange(-1, 1.01, 0.01)
yy = psyfxn(expt_params, xx)
ax.scatter(expt_results["coherence"], expt_results["prob_right"], color='k', linewidth=2)
ax.plot(xx, yy, 'k-', linewidth=2)
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
plt.show()


#### Calculate and plot both PSE and JND


In [ ]:
# calculate PSE
pse = float(ipsyfxn(expt_params, 0.5))
print(pse)

# calculate JND
jnd = float(expt_params[1] / np.sqrt(2))
print(jnd)

# plot the PSE +/- JND along with the data and fitted curve
fig, ax = plt.subplots()
ax.scatter(expt_results["coherence"], expt_results["prob_right"], color='k', linewidth=2, label='data')
ax.plot(xx, yy, 'k-', linewidth=2, label='fit')
ax.plot([pse, pse], [0, 0.5], 'r-', linewidth=1.5, label='pse')
jnd_x = np.array([pse - jnd, pse + jnd, pse + jnd, pse - jnd])
jnd_y = np.array([0, 0, 0.5, 0.5])
ax.fill(jnd_x, jnd_y, color='r', alpha=0.2, label='jnd')
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
ax.legend(loc='upper center', ncol=4)
plt.show()


**What do you see?** What do you expect to happen to this curve if you perform the experiment while being adapted to motion to the right?  What about to the left?

## Part B. Motion Adaptation Experiment

### Step 1. Data collection

To collect data, open the PsychoPy ".psyexp" file in the folder titled "motion-adaptation", and run that experiment. The instructions will appear on the screen before the trials begin. A single run of the experiment includes 140 trials.

### Step 2. Load your data

#### Find files for specific participant ID

Before running this section of code, enter the participant ID you used below.


In [ ]:
participant_id = "demo"

# check current working directory and find files
file_info = find_participant_files(Path("motion-adaptation") / "data", participant_id)
n_files = len(file_info)


**Confirm that your data file is listed above.** There should be one file.


In [ ]:
# load data and stack it into one table
adapt_data = load_psychopy_csvs(file_info, remove_instruction_rows=False)
adapt_data = adapt_data[adapt_data["coherence"].notna()].reset_index(drop=True)

# display data table size
n_trials = len(adapt_data)
print(f"number of trials = {n_trials}")


### Step 3. Analyze your data

For this experiment, the stimulus was a field of dots. Both direction of motion and motion coherence varied. The coherence defines the proportion of dots that move in the direction of motion. The direction of motion was either to the right (0 degrees) or to the left (180 degrees), and the task was to report whether the motion was rightward or leftward.

For each trial, PsychoPy saved the motion direction, the motion coherence, and the participant's response. The responses were stored as 'left' or 'right'.


In [ ]:
# stash the stimulus location and response data
motion_coherence = adapt_data["coherence"].astype(float)
motion_direction = adapt_data["direction"].astype(float).copy()
resp_key = adapt_data["key_resp.keys"].astype(str)

# convert stimulus motion direction to 1 for right and -1 for left
motion_direction = motion_direction.replace({180: -1, 0: 1})

# create signed motion coherence
signed_motion_coherence = motion_coherence * motion_direction

# convert the response key to 1 for right and -1 for left
# (to match stimulus location)
resp_direction = np.where(resp_key.eq("right"), 1, -1)


#### Determine whether responses were correct or incorrect

For a response to be correct, the motion direction must match the response direction.


In [ ]:
# determine whether responses were correct or incorrect
resp_correct = motion_direction.to_numpy() == resp_direction

chose_right = resp_direction == 1

# display overall performance
print(f"overall percentage correct = {np.mean(resp_correct) * 100:.2f}%")


#### Determine performance per condition

We need to now separate by coherence and direction to visualize performance in a psychometric curve.


In [ ]:
# average across responses per contrast level
analysis_table = pd.DataFrame({"response": chose_right, "coherence": signed_motion_coherence})
adapt_results = (
    analysis_table
    .groupby("coherence", as_index=False)
    .agg(n_trials=("response", "size"), prob_right=("response", "mean"))
)

# stash stimulus levels & counts
coherence_levels = np.sort(adapt_results["coherence"].unique())
n_coherence = len(coherence_levels)

# display results
display(adapt_results)


#### Plot performance per condition


In [ ]:
fig, ax = plt.subplots()
ax.scatter(adapt_results["coherence"], adapt_results["prob_right"], color='k', linewidth=2)
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
plt.show()


#### **Fit data**

Fit the adaptation data with the same SciPy likelihood. The MATLAB adaptation fit uses tighter upper bounds of `0.1` for both $\gamma$ and $\lambda$, so this call preserves those bounds.


In [ ]:
adapt_scipy_result = fit_psychometric_scipy(
    x=adapt_results["coherence"].to_numpy(dtype=float),
    k=(adapt_results["prob_right"] * adapt_results["n_trials"]).to_numpy(dtype=float),
    n=adapt_results["n_trials"].to_numpy(dtype=float),
    initial_params=[0, 5, 0.1, 0.1],
    bounds=[(0, 180), (0, 100), (0, 0.1), (0, 0.1)],
)
adapt_params = adapt_scipy_result.x
adapt_param_table = display_fit_table(adapt_params)
display(adapt_param_table)
print(f"negative log-likelihood = {adapt_scipy_result.fun:.6f}")


#### Plot fit with data


In [ ]:
fig, ax = plt.subplots()
xx = np.arange(-1, 1.01, 0.01)
yy = psyfxn(adapt_params, xx)
ax.scatter(adapt_results["coherence"], adapt_results["prob_right"], color='k', linewidth=2)
ax.plot(xx, yy, 'k-', linewidth=2)
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
plt.show()


#### Calculate and plot both PSE and JND


In [ ]:
# calculate PSE
pse = float(ipsyfxn(adapt_params, 0.5))
print(pse)

# calculate JND
jnd = float(adapt_params[1] / np.sqrt(2))
print(jnd)

# plot the PSE +/- JND along with the data and fitted curve
fig, ax = plt.subplots()
ax.scatter(adapt_results["coherence"], adapt_results["prob_right"], color='k', linewidth=2, label='data')
ax.plot(xx, yy, 'k-', linewidth=2, label='fit')
ax.plot([pse, pse], [0, 0.5], 'r-', linewidth=1.5, label='pse')
jnd_x = np.array([pse - jnd, pse + jnd, pse + jnd, pse - jnd])
jnd_y = np.array([0, 0, 0.5, 0.5])
ax.fill(jnd_x, jnd_y, color='r', alpha=0.2, label='jnd')
ax.set_xlabel("signed coherence")
ax.set_ylabel("probability of choosing right")
ax.legend(loc='upper center', ncol=4)
plt.show()
